# ZETTY RBA — Google Colab 전용 학습
Mac 학습은 중단되어 있습니다. **Google 호스팅 런타임**을 연결하세요. 로컬 런타임을 선택하지 마세요. 이 노트북은 CPU 모델을 실행하며 GPU가 필요하지 않습니다.

완료된 Online Shop/RBD 179개 실험은 다시 실행하지 않습니다. `resume`은 RBA validation 완료 모델 11개를 재사용하고 중단된 RF부터 다시 수행합니다. `fresh`는 RBA만 새로 전처리·학습합니다. 무료 Colab의 RAM/연속 실행 시간은 보장되지 않습니다.


## 1. 원격 runtime / Python 확인
Python 3.12가 아니면 진행하지 않습니다. Drive 승인은 이후 셀에서 수행합니다.


In [ ]:
import os, sys, platform
assert platform.system() == 'Linux' and os.environ.get('COLAB_RELEASE_TAG'), 'Google-hosted Colab required; do not use Mac local runtime'
assert sys.version_info[:2] == (3, 12), f'Python 3.12 required, got {sys.version}'
print(platform.platform(), sys.version)


## 2. 코드 및 고정 라이브러리 설치
설치 뒤 NumPy/scikit-learn을 이미 import한 세션이라면 런타임을 다시 시작하고 1번부터 실행하세요.


In [ ]:
import subprocess
from pathlib import Path
REPO = Path('/content/zetty-uba')
REF = 'feature/public-data-training-star'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch', 'https://github.com/jjyj0203/uba-analyzer.git', str(REPO)], check=True)
print('Code commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', '-r', str(REPO/'requirements-ml.lock')], check=True)


## 3. Drive 연결 및 모드 선택
`resume`: Mac의 `ml-runs/20260927-full`에서 **prepared/rba와 runs/rba**를 같은 구조로 Drive의 `zetty/20260927-full`에 업로드합니다. 약 2.7 GB의 배열과 모델/분할 파일이 필요합니다. `source_snapshot`을 빠뜨리지 마세요.

`fresh`: 업로드 없이 공식 원자료를 Colab에서 받습니다. Mac에서 끝낸 RBA 11개 후보도 다시 실행하는 새 연구 버전입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODE = 'resume'  # or 'fresh'
CHECKPOINT = Path('/content/drive/MyDrive/zetty/20260927-full')
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUTPUT = Path('/content') / ('zetty-rba-' + RUN_ID)
BACKUP = Path('/content/drive/MyDrive/zetty/colab-runs') / RUN_ID
print('Mode:', MODE, '\nLocal VM:', OUTPUT, '\nDrive backup:', BACKUP)


## 4. 지금까지 학습한 모델 확인 — 학습 실행 없음
EVALUATED는 최종 test까지 완료, VALIDATED는 학습·보정·validation만 완료입니다. RBA의 최종 test는 아직 없습니다.


In [ ]:
if MODE == 'resume':
    assert (CHECKPOINT/'prepared/rba/dataset.json').is_file(), 'Upload prepared/rba first'
    assert (CHECKPOINT/'runs/rba/source_snapshot/study/runner.py').is_file(), 'Upload runs/rba including source_snapshot'
    subprocess.run([sys.executable, str(REPO/'scripts/ml/inspect_study.py'), str(CHECKPOINT/'runs/rba')], check=True)
else:
    print('Fresh mode: original Mac results remain unchanged. See docs/ml/07-full-data-study/comparison.csv')


## 5. Colab에서 학습 실행
Drive 자료를 VM 로컬 디스크로 복사하고 학습합니다. 완료 artifact를 30초 간격으로 백업합니다. 재개 시 원본 Mac run을 수정하지 않고 새 출력 디렉터리를 사용합니다. RF가 중간에 끊기면 RF fit은 처음부터 다시 수행합니다.


In [ ]:
command = [sys.executable, str(REPO/'scripts/ml/colab_rba.py'), '--mode', MODE, '--output', str(OUTPUT), '--backup', str(BACKUP)]
if MODE == 'resume':
    command += ['--checkpoint', str(CHECKPOINT)]
subprocess.run(command, check=True)


## 6. 완료 상태·최종 성능 확인
최종 test를 보고 모델을 재선택하지 않습니다. selection은 validation만으로 고정됩니다. COMPLETED일 때만 추론 환경에 반영합니다.


In [ ]:
import json
manifest = json.loads((OUTPUT/'runs/rba/study.json').read_text())
print('Status:', manifest['status'])
print('Selection:', manifest.get('selection'))
subprocess.run([sys.executable, str(REPO/'scripts/ml/inspect_study.py'), str(OUTPUT/'runs/rba')], check=True)
assert manifest['status'] == 'COMPLETED', 'Not ready for inference deployment'


## 7. 결과 내려받기
ZIP에는 모델, threshold/feature manifest, source snapshot, split과 cloud 실행 기록을 담습니다. Mac에서 새 버전 디렉터리에 풀고 study.json SHA256을 새로 고정해 lab Compose 모델 경로를 변경하세요. 기존 RBD 모델을 덮어쓰지 않습니다.


In [ ]:
import zipfile
archive = OUTPUT/'zetty-rba-completed.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED) as z:
    for path in (OUTPUT/'runs/rba').rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and path.suffix != '.tmp':
            z.write(path, path.relative_to(OUTPUT/'runs'))
from google.colab import files
files.download(str(archive))
print('Persistent Drive copy:', BACKUP/'runs/rba')


## 중단·재개
Colab이 종료되면 Drive BACKUP을 다음 실행의 CHECKPOINT로 지정하고 `resume`으로 실행합니다. 원본 배열 백업이 끝나기 전 종료되면 최초 Mac checkpoint 또는 `fresh`가 필요합니다. 작업이 끝나면 **런타임 연결 해제 및 삭제**로 자원을 반환하세요.

이 notebook JSON/구문은 로컬에서 확인했지만 Google 계정 로그인·Drive 업로드·원격 학습은 사용자가 실행해야 합니다. [Colab FAQ](https://research.google.com/colaboratory/faq.html)
